# US Road Paving and Resurfacing: Data Acquisition

Pulls annual paving and resurfacing activity from two federal sources:

1. FHWA Highway Statistics Series (SF tables) — state highway capital outlay
  by improvement type, in dollars and miles.
2. FHWA HPMS public release — segment-level Year of Last Improvement,
  aggregated to miles per year.

Outputs tidy CSVs into ./data/processed/.

Caveat retained throughout: neither source captures the full universe of
locally owned road mileage. Treat results as a well-instrumented estimate.

In [1]:
# Run once. geopandas/pyogrio are only needed for the HPMS section.
%pip install -q requests beautifulsoup4 pandas openpyxl xlrd tqdm
%pip install -q geopandas pyogrio  # heavy; skip if you only want the SF tables

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re, time, zipfile, hashlib
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

RAW  = Path("data/raw");       RAW.mkdir(parents=True, exist_ok=True)
PROC = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)

FHWA_STATS = "https://www.fhwa.dot.gov/policyinformation/statistics.cfm"
HPMS_PAGE  = "https://www.fhwa.dot.gov/policyinformation/hpms/shapefiles.cfm"

YEARS = range(2010, 2024)

SESSION = requests.Session()
SESSION.headers.update({
   "User-Agent": "research-notebook/1.0 (contact: e.schnitger@icloud.com)"
})

In [3]:
def fetch(url, dest_dir=RAW, filename=None, pause=1.0):
   """Download once, reuse thereafter. Returns a local Path."""
   filename = filename or (url.split("/")[-1].split("?")[0] or
                           hashlib.md5(url.encode()).hexdigest())
   dest = Path(dest_dir) / filename
   if dest.exists() and dest.stat().st_size > 0:
       return dest
   dest.parent.mkdir(parents=True, exist_ok=True)
   with SESSION.get(url, stream=True, timeout=120) as r:
       r.raise_for_status()
       total = int(r.headers.get("content-length", 0)) or None
       with open(dest, "wb") as f, tqdm(total=total, unit="B",
                                        unit_scale=True, desc=filename) as bar:
           for chunk in r.iter_content(1 << 16):
               f.write(chunk); bar.update(len(chunk))
   time.sleep(pause)   # FHWA is a public good; do not hammer it
   return dest


def soup(url):
   r = SESSION.get(url, timeout=60); r.raise_for_status()
   return BeautifulSoup(r.text, "html.parser")

In [4]:
def find_year_pages():
    s = soup(FHWA_STATS)
    out = {}
    for opt in s.find_all("option"):
        text = opt.get_text(" ", strip=True)
        val = opt.get("value", "")
        m = re.search(r"(19|20)\d{2}", text)
        if m and val and val != "#":
            out[int(m.group(0))] = urljoin(FHWA_STATS, val)
    return dict(sorted(out.items()))

year_pages = find_year_pages()
print(f"Found {len(year_pages)} year pages")

Found 31 year pages


In [5]:
from pathlib import Path

WANTED = re.compile(r"\bsf-?12[ab]?\b", re.I)

def find_sf_tables(year_url):
    s = soup(year_url)
    hits = []
    for a in s.find_all("a", href=True):
        if a.get_text(" ", strip=True) != "Excel":
            continue  # skip PDF links and everything else; one hit per table is enough
        href = a["href"]
        stem = Path(href).stem  # e.g. "sf12a" from "xls/sf12a.xlsx"
        if WANTED.search(stem):
            # caption text lives outside the <a>; grab it from the enclosing row if present
            row = a.find_parent(["li", "tr", "p", "div"])
            caption = row.get_text(" ", strip=True) if row else stem
            hits.append({"label": caption, "code": stem, "url": urljoin(year_url, href)})
    return hits

In [6]:
catalog_rows = []
for yr, url in tqdm(year_pages.items(), desc="Scanning year pages"):
    if yr not in YEARS:
        continue
    for h in find_sf_tables(url):
        catalog_rows.append({"year": yr, **h})

catalog = pd.DataFrame(catalog_rows)
#print(f"catalog: {len(catalog)} rows across {catalog['year'].nunique() if len(catalog) else 0} years")
#print(catalog.head())

Scanning year pages:   0%|          | 0/31 [00:00<?, ?it/s]

In [7]:
import openpyxl
import xlrd

def get_merged_value_map(path, sheet_name=0):
    """Return {(row, col): value} for every cell inside a merged range,
    using the merge's actual top-left value — no ffill heuristics needed.
    Handles both .xlsx (openpyxl) and legacy .xls (xlrd)."""
    suffix = Path(path).suffix.lower()
    fill_map = {}

    if suffix == ".xls":
        wb = xlrd.open_workbook(path)
        sheet = wb.sheet_by_index(sheet_name) if isinstance(sheet_name, int) else wb.sheet_by_name(sheet_name)
        for rlo, rhi, clo, chi in sheet.merged_cells:
            top_left = sheet.cell_value(rlo, clo)
            for r in range(rlo, rhi):
                for c in range(clo, chi):
                    fill_map[(r, c)] = top_left  # xlrd is already 0-indexed
    else:
        wb = openpyxl.load_workbook(path, data_only=True)
        ws = wb[wb.sheetnames[sheet_name]] if isinstance(sheet_name, int) else wb[sheet_name]
        for merged_range in ws.merged_cells.ranges:
            top_left = ws.cell(merged_range.min_row, merged_range.min_col).value
            for r in range(merged_range.min_row, merged_range.max_row + 1):
                for c in range(merged_range.min_col, merged_range.max_col + 1):
                    fill_map[(r - 1, c - 1)] = top_left  # convert to 0-indexed to match pandas

    return fill_map

In [8]:
def read_sf_table(path):
    """Highway Statistics workbooks carry multi-row headers and inline
    footnotes. Strategy: read raw, find the row containing the state names
    column, treat everything above as header debris."""
    raw = pd.read_excel(path, header=None, dtype=object)

    state_row = None
    for i in range(min(25, len(raw))):
        row = raw.iloc[i].astype(str).str.upper()
        if row.str.contains("ALABAMA").any():
            state_row = i
            break
    if state_row is None:
        raise ValueError(f"Could not locate data start in {path.name}")

    merge_map = get_merged_value_map(path)
    header_rows = raw.iloc[max(0, state_row - 4):state_row].copy()
    for r_idx in header_rows.index:
        for c_idx in header_rows.columns:
            if pd.isna(header_rows.at[r_idx, c_idx]) and (r_idx, c_idx) in merge_map:
                header_rows.at[r_idx, c_idx] = merge_map[(r_idx, c_idx)]
    header_rows = header_rows.fillna("").astype(str)
    header = header_rows.agg(" ".join, axis=0).str.strip().str.replace(r"\s+", " ", regex=True)

    df = raw.iloc[state_row:].copy()
    df.columns = [c if c else f"col_{i}" for i, c in enumerate(header)]
    df = df.rename(columns={df.columns[0]: "state"})

    df["state"] = df["state"].astype(str).str.strip()
    df = df[~df["state"].str.match(r"^(nan|Total|\d|\s*$)", case=False, na=True)]
    return df.reset_index(drop=True)

In [9]:
frames = {}
for row in catalog.itertuples():
    p = fetch(row.url, dest_dir=RAW / "highway_statistics",
              filename=f"{row.year}_{Path(row.url).name}")
    try:
        frames[(row.year, row.code)] = read_sf_table(p)
        print(f"{row.year} {row.code}: {frames[(row.year, row.code)].shape}")
    except Exception as e:
        print(f"{row.year} {row.code}: FAILED — {e}")

for (yr, code), df in frames.items():
    dupes = df.columns[df.columns.duplicated()].tolist()
    if dupes:
        print(yr, code, "duplicated columns:", dupes)

data/raw/highway_statistics/2011_sf12.xlsx
2011 sf12: (52, 13)
data/raw/highway_statistics/2012_sf12.xls
2012 sf12: (52, 13)
data/raw/highway_statistics/2012_sf12a.xls
2012 sf12a: (52, 19)
data/raw/highway_statistics/2012_sf12b.xls
2012 sf12b: (52, 15)
data/raw/highway_statistics/2013_sf12.xls
2013 sf12: (52, 13)
data/raw/highway_statistics/2013_sf12a.xls
2013 sf12a: (52, 19)
data/raw/highway_statistics/2013_sf12b.xls
2013 sf12b: (52, 15)
data/raw/highway_statistics/2014_sf12.xls
2014 sf12: (52, 13)
data/raw/highway_statistics/2014_sf12a.xls
2014 sf12a: (52, 19)
data/raw/highway_statistics/2014_sf12b.xls
2014 sf12b: (52, 15)
data/raw/highway_statistics/2015_sf12.xlsx
2015 sf12: (52, 21)
data/raw/highway_statistics/2015_sf12a.xlsx
2015 sf12a: (52, 18)
data/raw/highway_statistics/2015_sf12b.xlsx
2015 sf12b: (53, 15)
data/raw/highway_statistics/2016_sf12.xlsx
2016 sf12: (52, 21)
data/raw/highway_statistics/2016_sf12a.xlsx
2016 sf12a: (52, 18)
data/raw/highway_statistics/2016_sf12b.xlsx
20

In [14]:
RESURF = re.compile(r"resurfac|restorat|rehabilit|3R|4R", re.I)

records = []
for (yr, code), df in frames.items():
    if code != "sf12a":
        continue
    for col in df.columns:
        if col == "state":
            continue
        records.append(pd.DataFrame({
            "year": yr,
            "code": code,
            "state": df["state"],
            "measure": str(col),
            "dollars_thousands": pd.to_numeric(df[col], errors="coerce"),
            "is_resurfacing": bool(RESURF.search(str(col))),
        }))

tidy = pd.concat(records, ignore_index=True).dropna(subset=["dollars_thousands"])
national_dollars = (tidy[tidy.is_resurfacing]
                     .groupby("year", as_index=False)["dollars_thousands"].sum())

In [11]:
for (yr, code), df in frames.items():
    if code == "sf12a":
        dupes = df.columns[df.columns.duplicated()].tolist()
        if dupes:
            print(yr, code, dupes)

In [12]:
def find_hpms_links():
   s = soup(HPMS_PAGE)
   return [{"label": a.get_text(" ", strip=True),
            "url": urljoin(HPMS_PAGE, a["href"])}
           for a in s.find_all("a", href=True)
           if a["href"].lower().endswith(".zip")]

hpms_links = pd.DataFrame(find_hpms_links())
hpms_links.head(20)

""


In [13]:
import geopandas as gpd

def hpms_miles_by_improvement_year(zip_path):
   """HPMS public release is normalised long-format: one row per data item
   per linear-referenced segment. Filter, then weight by segment length."""
   with zipfile.ZipFile(zip_path) as z:
       z.extractall(zip_path.parent / zip_path.stem)
   root = zip_path.parent / zip_path.stem

   src = next((p for p in root.rglob("*.gdb")), None) or \
         next(root.rglob("*.shp"))

   gdf = gpd.read_file(src, engine="pyogrio")
   gdf.columns = [c.upper() for c in gdf.columns]

   # Column names vary by vintage; probe rather than assume.
   item_col  = next(c for c in gdf.columns if "DATA_ITEM" in c)
   value_col = next(c for c in gdf.columns if "VALUE_NUMERIC" in c)
   len_col   = next(c for c in gdf.columns
                    if c in ("END_POINT", "SECTION_LENGTH", "MILES"))

   sub = gdf[gdf[item_col].astype(str).str.contains("YEAR_LAST_IMPROVEMENT",
                                                    case=False, na=False)]
   if len_col == "END_POINT":
       sub = sub.assign(SEG_MILES=sub["END_POINT"] - sub["BEGIN_POINT"])
   else:
       sub = sub.assign(SEG_MILES=sub[len_col])

   return (sub.groupby(sub[value_col].astype("Int64"), as_index=False)
              ["SEG_MILES"].sum().rename(columns={value_col: "improvement_year", "SEG_MILES": "miles"}))

In [16]:
r2012 = tidy[(tidy.year == 2012) & tidy.is_resurfacing][["state", "measure", "dollars_thousands"]].sort_values("state")
r2013 = tidy[(tidy.year == 2013) & tidy.is_resurfacing][["state", "measure", "dollars_thousands"]].sort_values("state")

print(r2012.measure.unique())   # which column(s) RESURF actually matched
print(r2013.measure.unique())

merged = r2012.merge(r2013, on="state", suffixes=("_2012", "_2013"))
print((merged.dollars_thousands_2012 - merged.dollars_thousands_2013).abs().sum())
print(merged.head(10))

<ArrowStringArray>
['3R MINOR WIDENING', 'RESTORATION & REHABILITATION', 'RESURFACING']
Length: 3, dtype: str
<ArrowStringArray>
['3R MINOR WIDENING', 'RESTORATION & REHABILITATION', 'RESURFACING']
Length: 3, dtype: str
6562612.5961614
     state                  measure_2012  dollars_thousands_2012  \
0  Alabama             3R MINOR WIDENING                 0.00000   
1  Alabama             3R MINOR WIDENING                 0.00000   
2  Alabama             3R MINOR WIDENING                 0.00000   
3  Alabama  RESTORATION & REHABILITATION             47263.31400   
4  Alabama  RESTORATION & REHABILITATION             47263.31400   
5  Alabama  RESTORATION & REHABILITATION             47263.31400   
6  Alabama                   RESURFACING                 0.00000   
7  Alabama                   RESURFACING                 0.00000   
8  Alabama                   RESURFACING                 0.00000   
9   Alaska             3R MINOR WIDENING             16951.82485   

               